In [ ]:
!pip install transformers torch pandas -q

In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import re

ANSI_RESET = "\033[0m"
ANSI_BOLD = "\033[1m"
ANSI_CYAN = "\033[96m"
ANSI_GREEN = "\033[92m"
ANSI_YELLOW = "\033[93m"

print("=" * 70)
print(f"{ANSI_BOLD}Loading Model & Processing Consolidated News Entities{ANSI_RESET}")
print("=" * 70)

MODEL_NAME = "dslim/bert-base-NER"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

df = pd.read_csv("5000news.csv")
titles = df['Title'].fillna("").tolist()

# Financial & News noise words to ignore
STOP_WORDS = {
    "q1", "q2", "q3", "q4", "october", "oct", "november", "december", "january",
    "february", "march", "april", "may", "june", "july", "august", "september",
    "chips start", "transcript", "stocks", "ai", "buy", "top"
}

def clean_word(text):
    cleaned = re.sub(r'^[#\s\.\,\-\']+|[#\s\.\,\-\']+$', '', text)
    return cleaned.strip()

print("Extracting entities into a single column...")
batch_results = ner_pipeline(titles, batch_size=32)

consolidated_entities = []

for entities in batch_results:
    extracted_set = set()

    for entity in entities:
        score = entity.get('score', 1.0)
        word = clean_word(entity['word'])
        group = entity['entity_group']

        # Filter low confidence & junk/noise words
        if score < 0.60 or len(word) <= 2 or word.lower() in STOP_WORDS:
            continue

        if group in ["ORG", "PER", "LOC"]:
            # Format: Entity (TYPE) -> e.g. Intel (ORG), Ken Griffin (PER)
            extracted_set.add(f"{word} ({group})")

    if extracted_set:
        # Sort and join all entities into a single string
        consolidated_entities.append(", ".join(sorted(extracted_set)))
    else:
        consolidated_entities.append("None")

# Replace separate columns with one single clean column
df['Extracted_Entities'] = consolidated_entities

print(f"{ANSI_GREEN}Extraction Complete! All entities merged into 'Extracted_Entities'.{ANSI_RESET}\n")

print("=" * 70)
print(f"{ANSI_BOLD}Previewing Clean Dataset (First 10 Rows){ANSI_RESET}")
print("=" * 70)

for idx, row in df.head(10).iterrows():
    print(f"{ANSI_CYAN}Title:{ANSI_RESET} {row['Title']}")
    print(f"  📌 {ANSI_YELLOW}Entities:{ANSI_RESET} {row['Extracted_Entities']}")
    print("-" * 70)

# Save Clean Consolidated CSV
output_filename = "5000news_single_column_ner.csv"
df.to_csv(output_filename, index=False)

# Auto Download in Colab
from google.colab import files
files.download(output_filename)

Loading Model & Processing Consolidated News Entities


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting entities into a single column...
Extraction Complete! All entities merged into 'Extracted_Entities'.

Previewing Clean Dataset (First 10 Rows)
Title: Vietnam Airlines data leak exposes a crisis of transparency
  📌 Entities: Vietnam Airlines (ORG)
----------------------------------------------------------------------
Title: DOJ seizes $15 billion bitcoin in SE Asia crypto scam bust
  📌 Entities: DOJ (ORG), SE Asia (LOC)
----------------------------------------------------------------------
Title: Think It's Too Late to Buy IonQ? Here's the 1 Reason Why There's Still Time
  📌 Entities: None
----------------------------------------------------------------------
Title: Intel's Crucial Panther Lake Chips Start Production
  📌 Entities: Intel (ORG)
----------------------------------------------------------------------
Title: 3 Robotics Stocks to Buy in October
  📌 Entities: None
----------------------------------------------------------------------
Title: Think It's Too Late to Buy

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>